Inverter Division (ELM) - 725 / 730 Wp
=
Quick sizing calculations for splitting the strings of a plant between its inverters, with two module power bins (725 Wp and 730 Wp, 32 modules per string). The plant size (464 inverters in 29 PSTs, 12367 strings) is the same as PV Paiva; the Paiva notebook repeats these checks for the 735 / 740 Wp modules.

**Workflow overview:**
1. Inverters per PST.
2. Max strings per MPPT / per inverter from the MPPT current limits, and the average strings per inverter.
3. Max modules per string from the inverter max input voltage.
4. How many inverters of each module bin get 26 or 27 strings.
5. Helper: combinations of 27- and 28-string inverters that add up to a target number of strings.

**Input prerequisites:** no files are read or written. Everything is hardcoded in the first cell: module data per bin (`mod1`, `mod2`: power, module quantity, string count, Vmp, Voc, Imp, Isc), inverter limits (`inv`), and the inverter and PST counts.

Inputs and inverters per PST
=
- `mod1` (725 Wp) and `mod2` (730 Wp): module quantity, string count and datasheet values (Vmp, Voc, Imp, Isc) of each bin.
- `inv`: inverter limits (max input voltage 1500 V, 6 MPPTs, 106 A max current and 180 A max short circuit per MPPT, 6 inputs per MPPT).
- `inverters / PSTs` gives inverters per PST. If it is not an integer, prints the two neighbours (`ceil` / `floor`). Here 464 / 29 = 16 exactly.

In [25]:
import math

mod1 = {
    'Potência': 725,
    'Qtd.': 140832,
    'Strings': 4401,
    'Vmp': 41,
    'Voc': 49.2,
    'Imp': 17.69,
    'Isc': 18.74
}

mod2 = {
    'Potência': 730,
    'Qtd.': 254912,
    'Strings': 7966,
    'Vmp': 41.11,
    'Voc': 49.36,
    'Imp': 17.76,
    'Isc': 18.81
}

inv = {
    'Max Input Voltage': 1500,
    'No. of MPPTs': 6,
    'Max Current per MPPT': 106,
    'Max Short Circuit per MPPT': 180,
    'Max PV Inputs per MPPT': 6
}

inverters = 464
PSTs = 29
inverters_per_PST = inverters / PSTs
if inverters_per_PST % 1 != 0:
    inverter_per_PST_up = math.ceil(inverters_per_PST)
    inverter_per_PST_down = math.floor(inverters_per_PST)
    print('Inverters per PST is not a multiple of 1.')
    print(f'The number of inverters per PSTs might be: {inverter_per_PST_up} and {inverter_per_PST_down}')
else:
    print(f'The number of inverters per PST might be: {inverters_per_PST}')

The number of inverters per PST might be: 16.0


Strings per MPPT and per inverter
=
- Loops over the number of strings in parallel on one MPPT and checks the MPPT current limits with the **730 Wp (`mod2`)** module (the higher-current bin): `n · Isc ≤ Max Short Circuit per MPPT` and `n · Imp ≤ Max Current per MPPT`.
- `max_number_of_PV` = the smallest of the two current limits and `Max PV Inputs per MPPT`. Strings per inverter = that × `No. of MPPTs`.
  - Warning: `range(6)` stops at 5, so 6 strings is never tested, and each iteration overwrites the previous one, so only the last `n = 5` really counts. The result (5) is right here only because 6 × 17.76 A is above 106 A anyway.
- `strings_per_inverter` = total strings / inverters. If it is not an integer, the split is between `ceil` and `ceil − 1` strings per inverter.
- The f-string with `inv['No. of MPPTs']` inside single quotes needs Python 3.12+.

In [26]:
for i in range(inv['Max PV Inputs per MPPT']):
    sc_current = i*mod2['Isc']
    mp_current = i*mod2['Imp']
    max_number_of_PV_ISC = i
    max_number_of_PV_MP = i

    if sc_current > inv['Max Short Circuit per MPPT']:
        sc_current = (i-1)*mod2['Isc']
        max_number_of_PV_ISC = (i-1)
    if mp_current > inv['Max Current per MPPT']:
        mp_current = (i-1)*mod2['Imp']
        max_number_of_PV_MP = (i-1)

max_number_of_PV = min(max_number_of_PV_MP, max_number_of_PV_ISC, inv['Max PV Inputs per MPPT'])

print(f'The max number of strings per MPPT is: {max_number_of_PV}')
print(f'The max number of strings per inverter is: {max_number_of_PV*inv['No. of MPPTs']}')

strings_per_inverter = (mod1['Strings']+mod2['Strings'])/inverters
strings_per_inverter_up = math.ceil(strings_per_inverter)
if strings_per_inverter_up != strings_per_inverter:
    print('\nStrings per inverter is not a multiple of 1.')
    print(f'The number of strings per inverter might be: {strings_per_inverter_up} and {strings_per_inverter_up-1}')
else:
    print(f'The number of strings per inverter might be: {strings_per_inverter}')

The max number of strings per MPPT is: 5
The max number of strings per inverter is: 30

Strings per inverter is not a multiple of 1.
The number of strings per inverter might be: 27 and 26


Max modules per string
=
- Increases the number of modules until `n · Voc` goes over `Max Input Voltage`, and keeps the previous value (30 with the 730 Wp Voc). It uses Voc at STC, with no low-temperature correction.
- The result is then overwritten with the hardcoded `modules_per_inverter = 32` (the name means modules **per string**), so the loop has no effect.
  - Warning: 32 × 49.36 V = 1579.5 V, above the 1500 V limit even at STC. The hardcoded 32 matches the module quantities (`Qtd. / Strings` = 32 for both bins), so the voltage check disagrees with the design.

In [27]:
i = 1
while True:
    string_voltage = i * mod2['Voc']
    if string_voltage > inv['Max Input Voltage']:
        string_voltage = (i-1)*mod2['Voc']
        modules_per_inverter = i-1
        break
    i += 1

modules_per_inverter = 32
print(f'The max number of modules per string is: {modules_per_inverter}')

The max number of modules per string is: 32


26 / 27 strings per inverter, per module bin
=
Brute-force search for how many inverters of each bin get 26 or 27 strings.
- For each bin, the total number of inverters is fixed: `ceil(bin strings / strings_per_inverter)`, minus 1 for the 725 Wp bin (165) and not for the 730 Wp bin (299). That `− 1` is what makes the two totals add up to 464.
- For each split `i` (26-string) / `total − i` (27-string), it checks that the strings add up (`condition1` / `condition3`) and that the modules add up (`condition2` / `condition4`). The module check is redundant here, because `Qtd.` is exactly `Strings × 32`.
- Prints the four counts only if all four conditions are true. If any bin has no exact split, it prints nothing, with no warning.

In [30]:
max_number_of_725_inverters = math.ceil(mod1['Strings']/strings_per_inverter)
for i in range(max_number_of_725_inverters):
    inv_26_725 = i
    inv_27_725 = max_number_of_725_inverters - i-1
    condition1 = inv_26_725*26 + inv_27_725*27 == mod1['Strings']
    condition2 = inv_26_725*26*modules_per_inverter + inv_27_725*27*modules_per_inverter == mod1['Qtd.']
    if condition1 and condition2:
        break

max_number_of_730_inverters = math.ceil(mod2['Strings']/strings_per_inverter)
for i in range(max_number_of_730_inverters):
    inv_26_730 = i
    inv_27_730 = max_number_of_730_inverters - i
    condition3 = inv_26_730*26 + inv_27_730*27 == mod2['Strings']
    condition4 = inv_26_730*26*modules_per_inverter + inv_27_730*27*modules_per_inverter == mod2['Qtd.']
    if condition3 and condition4:
        break

if condition1 and condition2 and condition3 and condition4:
    print(f'The number of inverters with module of 725Wp with 26 strings per inverter is: {inv_26_725}')
    print(f'The number of inverters with module of 725Wp with 27 strings per inverter is: {inv_27_725}')
    print(f'The number of inverters with module of 730Wp with 26 strings per inverter is: {inv_26_730}')
    print(f'The number of inverters with module of 730Wp with 27 strings per inverter is: {inv_27_730}')

The number of inverters with module of 725Wp with 26 strings per inverter is: 54
The number of inverters with module of 725Wp with 27 strings per inverter is: 111
The number of inverters with module of 730Wp with 26 strings per inverter is: 107
The number of inverters with module of 730Wp with 27 strings per inverter is: 192


Helper: 27- and 28-string combinations for a target
=
`find_complete_strings(target)` looks for `i` inverters with 27 strings and `j` with 28 that add up to `target` strings (`target = 932 − 9`, hardcoded).
- Exact matches: returns a list of every `(i, j, target)`.
- No exact match: searches outward (`target + 1`, `target − 1`, `target + 2`, ...) and returns the first `(i, j, value)` found, as a single tuple, not a list. The higher value is tried first.
- The commented prints still refer to ×28 / ×29, and the saved output `[(7, 7, 371)]` doesn't fit 27/28 (7·27 + 7·28 = 385): it is from an older version of the cell.
- `import math` is repeated from the first cell.

In [67]:
import math

target = 932-9

def find_complete_strings(target):
    result = []
    # Check exact match first
    for i in range(math.ceil(target/27) + 1):
        for j in range(math.ceil(target/28) + 1):
            if i*27 + j*28 == target:
                result.append((i,j,target))
    if result:
        return result

    # If no exact match, search outward
    offset = 1
    while True:
        # Check higher
        for i in range(math.ceil((target+offset)/27) + 1):
            for j in range(math.ceil((target+offset)/28) + 1):
                if i*27 + j*28 == target + offset:
                    return i, j, target + offset

        # Check lower (if positive)
        if target - offset >= 0:
            for i in range(math.ceil((target-offset)/27) + 1):
                for j in range(math.ceil((target-offset)/28) + 1):
                    if i*27 + j*28 == target - offset:
                        return i, j, target - offset

        offset += 1

result = find_complete_strings(target)
# print(f"The closest number to {target} that works is: {result[2]}")
# print(f"Using {result[0]}×28 + {result[1]}×29 = {result[2]}")
# print(f'The difference between {target} and {result[2]} is {abs(target - result[2])}')
print(result)

[(7, 7, 371)]
